In [1]:
import os
#import tomllib

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
from joblib import Parallel, delayed
from tqdm import tqdm

In [2]:
#def read_setting(setting_file_path: str) -> dict[str, Any]:
#    """設定ファイルを読み込む"""
#    with open(setting_file_path, mode="rb") as setting_file:
#        settings: dict[str, Any] = tomllib.load(setting_file)
#    return settings

def sinh_arcsinh_transform(z, eps, delta):
    """Y = sinh((asinh(z) + eps) / delta)を計算する"""
    return np.sinh((np.arcsinh(z) + eps) / delta)

def generate_2group_shash_random_values(
    eps,
    delta,
    sample_size_group1,
    sample_size_group2,
    n_simulation,
    seed
):
    """指定のパラメータをもつsinh-arcsinh分布から乱数を2群分生成する"""
    rng = np.random.default_rng(seed)
    z1 = rng.normal(loc=0, scale=1, size=(sample_size_group1, n_simulation)).astype(np.float32)
    z2 = rng.normal(loc=0, scale=1, size=(sample_size_group2, n_simulation)).astype(np.float32)
    group1 = sinh_arcsinh_transform(z1, eps, delta)
    group2 = sinh_arcsinh_transform(z2, eps, delta)
    return group1, group2

def t_test(group1, group2, method):
    """t検定を実行する"""
    if method == 'student':
        student_ttest = stats.ttest_ind(group1, group2, equal_var=True)
        return student_ttest
    elif method == 'welch':
        welch_ttest = stats.ttest_ind(group1, group2, equal_var=False)
        return welch_ttest
    else:
        raise ValueError(f"unknown method: {method!r}")
        
def calc_alpha_error_and_interval(p_values, alpha, ci_alpha, method = 'wilson'):
    """αエラーとその信頼区間を算出する"""
    p_values = p_values[~np.isnan(p_values)]
    reject_count = np.sum(p_values < alpha)
    sample_size = p_values.shape[0]
    alpha_error = reject_count / sample_size
    low, high = proportion_confint(reject_count, sample_size, alpha=ci_alpha, method=method)
    return alpha_error, low, high, sample_size

def run_one_cell(
    skewness,
    excess_kurtosis,
    eps,
    delta,
    sample_size_group1,
    sample_size_group2,
    significance_alpha,
    confidence_interval_alpha,
    quantiles,
    n_simulation,
    batch_size,
    seed
):
    """1つのセルのシミュレーションをバッチに分けて実行する"""
    # バッチ数とシードの設定
    n_batch = int(np.ceil(n_simulation / batch_size))
    batch_seeds = seed.spawn(n_batch)
    
    # バッチごとに計算を実行して統合
    student_pvals = []
    welch_pvals = []
    remaining = n_simulation
    for bi in range(n_batch):
        # 乱数生成
        n_this = min(batch_size, remaining)
        group1, group2 = generate_2group_shash_random_values(
            eps=eps,
            delta=delta,
            sample_size_group1=sample_size_group1,
            sample_size_group2=sample_size_group2,
            n_simulation=n_this,
            seed=batch_seeds[bi]
        )
        remaining -= n_this
        
        # t-検定
        student_ttest_result = t_test(group1, group2, method='student')
        welch_ttest_result = t_test(group1, group2, method='welch')
        
        # p値を保存
        student_pvals.append(student_ttest_result.pvalue)
        welch_pvals.append(welch_ttest_result.pvalue)
        
    # p値のリストを全バッチ結合
    student_pvals = np.concatenate(student_pvals)
    welch_pvals = np.concatenate(welch_pvals)
    
    # αエラーとその信頼区間を算出
    student_alpha_error, student_ci_low, student_ci_high, student_valid_n = calc_alpha_error_and_interval(student_pvals, significance_alpha, confidence_interval_alpha)
    welch_alpha_error, welch_ci_low, welch_ci_high, welch_valid_n = calc_alpha_error_and_interval(welch_pvals, significance_alpha, confidence_interval_alpha)
    
    # p値の分位点を算出する
    student_p_quantiles = np.nanquantile(student_pvals, quantiles)
    welch_p_quantiles = np.nanquantile(welch_pvals, quantiles)
    
    # 結果保存
    result = {
        'Skewness': skewness, 'Excess_kurtosis': excess_kurtosis, 'eps': eps, 'delta': delta,
        'SAMPLE_SIZE_G1': sample_size_group1, 'SAMPLE_SIZE_G2': sample_size_group2,
        'student_alpha_error': student_alpha_error, 'welch_alpha_error': welch_alpha_error,
        'student_ci_low': student_ci_low, 'student_ci_high': student_ci_high,
        'welch_ci_low': welch_ci_low, 'welch_ci_high': welch_ci_high,
        'student_valid_n': student_valid_n, 'welch_valid_n': welch_valid_n,
    }
    
    for q, p_value in zip(quantiles, student_p_quantiles):
        result[f'student_p_value_quantile_{q}'] = p_value

    for q, p_value in zip(quantiles, welch_p_quantiles):
        result[f'welch_p_value_quantile_{q}'] = p_value
    
    return result

In [3]:
def main():
    # 設定データ読み込み
    SIGNIFICANCE_ALPHA = 0.05
    CONFIDENCE_INTERVAL_ALPHA = 0.05
    QUANTILES = [0.01, 0.05, 0.10]
    N_SIMULATION = 1_000
    BATCH_SIZE = 100
    PARENT_SEED = 20260630
    N_JOBS = -1
    SAPLE_SIZE = [
        [15, 15], [25, 25], [50, 50],
        [7,  23], [12, 38], [25, 75],
        [3,  27], [5,  45], [10, 90]
    ]
    
    # sinh-arcsinh分布のパラメータを読み込み、必要なデータのみに絞り込み
    sash_data = pd.read_parquet('./sinh_arcsinh_params.parquet')
    sash_data = sash_data[sash_data['converged']==True].copy()
    sash_data = sash_data[['target_skewness', 'target_excess_kurtosis', 'eps', 'delta']].copy()
    
    # 全タスクをリスト化
    tasks = []
    seed_seq = np.random.SeedSequence(PARENT_SEED)
    for sample_size_g1, sample_size_g2 in SAPLE_SIZE:
        for skewness, excess_kurtosis, eps, delta in sash_data.values:
            seed = seed_seq.spawn(1)[0]
            tasks.append((skewness, excess_kurtosis, eps, delta, sample_size_g1, sample_size_g2, seed))
            
            break
        break
            
    # 並列実行 (ジェネレータを作ってからtqdmでラップして実行)
    sim_generator = Parallel(n_jobs=N_JOBS, return_as='generator_unordered')(
        delayed(run_one_cell)(
            skewness=skewness,
            excess_kurtosis=excess_kurtosis,
            eps=eps,
            delta=delta,
            sample_size_group1=sample_size_g1,
            sample_size_group2=sample_size_g2,
            significance_alpha=SIGNIFICANCE_ALPHA,
            confidence_interval_alpha=CONFIDENCE_INTERVAL_ALPHA,
            quantiles=QUANTILES,
            n_simulation=N_SIMULATION,
            batch_size=BATCH_SIZE,
            seed=seed
        )
        for (skewness, excess_kurtosis, eps, delta, sample_size_g1, sample_size_g2, seed) in tasks
    )
    results = list(tqdm(sim_generator, total=len(tasks)))
    
    # データフレームにして出力
    sim_result_data = pd.DataFrame(results)
    #os.makedirs('output', exist_ok=True)
    #sim_result_data.to_parquet('./output/alpha_error_sim_result.parquet', index=False)
    
    return sim_result_data

In [4]:
if __name__ == '__main__':
    sim_result_data = main()

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.01s/it]
